In [ ]:
import os, sys
from pathlib import Path

# Ensure project root is importable (handles running from /notebooks subfolder)
PROJECT_ROOT = Path.cwd()
if not (PROJECT_ROOT / "utils").exists() and (PROJECT_ROOT.parent / "utils").exists():
    PROJECT_ROOT = PROJECT_ROOT.parent
sys.path.insert(0, str(PROJECT_ROOT))

# Organization context
ORG_ID = os.environ.get("ORGANIZATION_ID", "").strip()
if not ORG_ID:
    raise ValueError("Set ORGANIZATION_ID env var before running notebooks.")

# Download latest CSVs from Storage into data_exports/{ORG_ID}/
from scripts.sync_storage import sync_org_exports
LOCAL_ORG_EXPORT_DIR = sync_org_exports(ORG_ID)  # returns Path("data_exports/<org_id>")
print("✅ Local org export folder:", LOCAL_ORG_EXPORT_DIR)


# 01_ingest_and_validate.ipynb
from utils.io_utils import should_run, save_state, mark_state_ran, write_output_json, read_csv_to_df
from datetime import datetime, timezone

DATA_DIR = str(LOCAL_ORG_EXPORT_DIR)
STATE_PATH = "state/01_state.json"
OUT_PATH = "outputs/admin/data_health.json"

input_files = [
    f"{DATA_DIR}/visits.csv",
    f"{DATA_DIR}/queue_events.csv",
    f"{DATA_DIR}/staff_service_log.csv",
    f"{DATA_DIR}/services.csv",
    f"{DATA_DIR}/counters.csv",
]

run, new_state = should_run(input_files, STATE_PATH)
if not run:
    write_output_json(OUT_PATH, {"updated": False})
    raise SystemExit("No new CSV changes detected.")

# Load
visits = read_csv_to_df(f"{DATA_DIR}/visits.csv", date_cols=["timestamp"])
events = read_csv_to_df(f"{DATA_DIR}/queue_events.csv", date_cols=["event_time"])
staff_log = read_csv_to_df(f"{DATA_DIR}/staff_service_log.csv", date_cols=["start_time", "end_time"])
services = read_csv_to_df(f"{DATA_DIR}/services.csv")
counters = read_csv_to_df(f"{DATA_DIR}/counters.csv")

# Validate schema (fail fast)
required_visits = {
    "visit_id","timestamp","dow","hour","is_weekend",
    "service_id","service_name","branch_id",
    "wait_time_minutes","service_time_minutes","status"
}
required_events = {"event_id","visit_id","event_time","event_type","staff_id","counter_id","service_id"}

missing_v = required_visits - set(visits.columns)
missing_e = required_events - set(events.columns)

if missing_v:
    raise ValueError(f"visits.csv missing columns: {sorted(missing_v)}")
if missing_e:
    raise ValueError(f"queue_events.csv missing columns: {sorted(missing_e)}")

health = {
  "updated": True,
  "generated_at": datetime.now(timezone.utc).isoformat().replace("+00:00","Z"),
  "rows": {
      "visits": int(len(visits)),
      "queue_events": int(len(events)),
      "staff_service_log": int(len(staff_log)),
      "services": int(len(services)),
      "counters": int(len(counters)),
  },
  "status_counts": visits["status"].value_counts(dropna=False).to_dict(),
  "event_type_counts": events["event_type"].value_counts(dropna=False).to_dict(),
  "null_rates": {
      "visits.branch_id": float(visits["branch_id"].isna().mean()),
      "queue_events.staff_id": float(events["staff_id"].isna().mean()),
      "queue_events.counter_id": float(events["counter_id"].isna().mean()),
  }
}

write_output_json(OUT_PATH, health)
save_state(STATE_PATH, mark_state_ran(new_state))

print("✅ 01 complete: outputs/admin/data_health.json written")


ModuleNotFoundError: No module named 'utils'